In [5]:
import numpy as np 
import gym
import pickle as pkl
import cv2


In [6]:

# ==========================================
# PARAMETERS
# ==========================================
EPSILON = 0.1
ALPHA = 0.1
GAMMA = 0.9
NUM_EPISODE = 500


In [7]:
# ==========================================
# VISUALIZATION FUNCTIONS
# ==========================================
def initialize_frame():
    width, height = 600, 200
    # FIX: Added dtype=np.uint8 and changed 255.0 to 255
    img = np.ones(shape=(height, width, 3), dtype=np.uint8) * 255
    margin_horizontal = 6
    margin_vertical = 2

    # Vertical Lines
    for i in range(13):
        img = cv2.line(img, (49 * i + margin_horizontal, margin_vertical),
                       (49 * i + margin_horizontal, 200 - margin_vertical), color=(0, 0, 0), thickness=1)

    # Horizontal Lines
    for i in range(5):
        img = cv2.line(img, (margin_horizontal, 49 * i + margin_vertical),
                       (600 - margin_horizontal, 49 * i + margin_vertical), color=(0, 0, 0), thickness=1)

    # Cliff Box
    img = cv2.rectangle(img, (49 * 1 + margin_horizontal + 2, 49 * 3 + margin_vertical + 2),
                        (49 * 11 + margin_horizontal - 2, 49 * 4 + margin_vertical - 2), color=(255, 0, 255),
                        thickness=-1)
    img = cv2.putText(img, text="Cliff", org=(49 * 5 + margin_horizontal, 49 * 4 + margin_vertical - 10),
                      fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(255, 255, 255), thickness=2)

    # Goal
    frame = cv2.putText(img, text="G", org=(49 * 11 + margin_horizontal + 10, 49 * 4 + margin_vertical - 10),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(0, 0, 0), thickness=2)
    return frame

def put_agent(img, state):
    margin_horizontal = 6
    margin_vertical = 2
    row, column = np.unravel_index(indices=state, shape=(4, 12))
    cv2.putText(img, text="A", org=(49 * column + margin_horizontal + 10, 49 * (row + 1) + margin_vertical - 10),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(0, 0, 0), thickness=2)
    return img




In [8]:

# ==========================================
# RL SETUP & POLICY
# ==========================================
cliffEnv = gym.make("CliffWalking-v0")

# Ensure "sarsa_q_table.pkl" is in the same directory as this script
with open("sarsa_q_table.pkl", "rb") as f:
    q_table = pkl.load(f)

def policy(state, explore=0.0):
    if np.random.random() <= explore:
        return int(np.random.randint(0, q_table.shape[1]))
    return int(np.argmax(q_table[state]))



In [10]:

# ==========================================
# MAIN LOOP
# ==========================================
for episode in range(NUM_EPISODE):
    done = False
    frame = initialize_frame()
    
    # Safely unpack reset() 
    reset_result = cliffEnv.reset()
    state = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    
    total_reward = 0
    episode_length = 0

    while not done:
        frame2 = put_agent(frame.copy(), state)
        cv2.imshow("Cliff Walking", frame2)
        cv2.waitKey(250)
        
        action = policy(state)
        
        # Safely unpack step() depending on Gym version
        step_result = cliffEnv.step(action)
        state = step_result[0]
        reward = step_result[1]
        
        # Check termination based on returned tuple length
        if len(step_result) == 5:
            done = step_result[2] or step_result[3]
        else:
            done = step_result[2]
        
        total_reward += reward
        episode_length += 1
    
    print(
        f"Episode --> {episode + 1} | "
        f"Episode Length --> {episode_length} | "
        f"Total Reward --> {total_reward}"
    )
        
cliffEnv.close()
cv2.destroyAllWindows()

Episode --> 1 | Episode Length --> 15 | Total Reward --> -15
Episode --> 2 | Episode Length --> 15 | Total Reward --> -15
Episode --> 3 | Episode Length --> 15 | Total Reward --> -15
Episode --> 4 | Episode Length --> 15 | Total Reward --> -15
Episode --> 5 | Episode Length --> 15 | Total Reward --> -15
Episode --> 6 | Episode Length --> 15 | Total Reward --> -15
Episode --> 7 | Episode Length --> 15 | Total Reward --> -15
Episode --> 8 | Episode Length --> 15 | Total Reward --> -15
Episode --> 9 | Episode Length --> 15 | Total Reward --> -15
Episode --> 10 | Episode Length --> 15 | Total Reward --> -15
Episode --> 11 | Episode Length --> 15 | Total Reward --> -15
Episode --> 12 | Episode Length --> 15 | Total Reward --> -15
Episode --> 13 | Episode Length --> 15 | Total Reward --> -15
Episode --> 14 | Episode Length --> 15 | Total Reward --> -15
Episode --> 15 | Episode Length --> 15 | Total Reward --> -15
Episode --> 16 | Episode Length --> 15 | Total Reward --> -15
Episode --> 17 | 

KeyboardInterrupt: 